<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.1. Wisconsin Cancer Classification - Neural Networks
  </strong>
</div>

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

raw_data = load_breast_cancer(as_frame=True)
df = raw_data.data
target = raw_data.target

In [ ]:
!pip install torch

In [ ]:
print(f"Forma del conjunto de datos: {df.shape}")
print(f"Distribución de clases: {pd.Series(target).value_counts()}")
print(f"Variables objetivo: {raw_data.target_names}")
print(f"Características: {raw_data.feature_names}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df, target, test_size=0.2, random_state=42, stratify=target
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_preds)
print(f"Precisión del modelo de Regresión Logística: {lr_accuracy:.4f}")
print("\nInforme de clasificación (Regresión Logística):")
print(classification_report(y_test, lr_preds))

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_train_tensor = torch.FloatTensor(y_train.values)
y_test_tensor = torch.FloatTensor(y_test.values)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
input_size = X_train_scaled.shape[1]
hidden_size1 = 64
hidden_size2 = 32
output_size = 1

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.layer2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.output_layer = nn.Linear(hidden_size2, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.output_layer(x)
        x = self.sigmoid(x)
        return x

In [ ]:
model = NeuralNetwork(input_size, hidden_size1, hidden_size2, output_size)
print(model)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 100
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

In [ ]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels.view(-1, 1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = (outputs.data > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted.view(-1) == labels).sum().item()

    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels.view(-1, 1))

            test_loss += loss.item()
            predicted = (outputs.data > 0.5).float()
            total_test += labels.size(0)
            correct_test += (predicted.view(-1) == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    avg_test_loss = test_loss / len(test_loader)
    train_accuracy = correct_train / total_train
    test_accuracy = correct_test / total_test

    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

    if (epoch + 1) % 10 == 0:
        print(
            f"Época [{epoch+1}/{num_epochs}], Pérdida Train: {avg_train_loss:.4f}, "
            f"Pérdida Test: {avg_test_loss:.4f}, Precisión Train: {train_accuracy:.4f}, "
            f"Precisión Test: {test_accuracy:.4f}"
        )

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_tensor = model(X_test_tensor)
    y_pred_prob = y_pred_tensor.numpy().flatten()
    y_pred = (y_pred_prob > 0.5).astype(int)

In [ ]:
nn_accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión del modelo de Red Neuronal: {nn_accuracy:.4f}")
print("\nInforme de clasificación (Red Neuronal):")
print(classification_report(y_test, y_pred))

In [ ]:
print("\nComparación de modelos:")
print(f"Precisión de Regresión Logística: {lr_accuracy:.4f}")
print(f"Precisión de Red Neuronal: {nn_accuracy:.4f}")

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Pérdida Entrenamiento")
plt.plot(test_losses, label="Pérdida Prueba")
plt.xlabel("Épocas")
plt.ylabel("Pérdida")
plt.title("Curvas de Pérdida")
plt.legend()

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Precisión Entrenamiento")
plt.plot(test_accuracies, label="Precisión Prueba")
plt.xlabel("Épocas")
plt.ylabel("Precisión")
plt.title("Curvas de Precisión")
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=raw_data.target_names,
    yticklabels=raw_data.target_names,
)
plt.xlabel("Predicción")
plt.ylabel("Valor Real")
plt.title("Matriz de Confusión - Red Neuronal")
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(10, 8))
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_probs)
lr_auc = auc(lr_fpr, lr_tpr)

In [ ]:
nn_fpr, nn_tpr, _ = roc_curve(y_test, y_pred_prob)
nn_auc = auc(nn_fpr, nn_tpr)

In [ ]:
plt.plot(lr_fpr, lr_tpr, label=f"Regresión Logística (AUC = {lr_auc:.3f})")
plt.plot(nn_fpr, nn_tpr, label=f"Red Neuronal (AUC = {nn_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Aleatorio")
plt.xlabel("Tasa de Falsos Positivos")
plt.ylabel("Tasa de Verdaderos Positivos")
plt.title("Curvas ROC - Comparación de Modelos")
plt.legend()
plt.show()
plt.close()

In [ ]:
def train_and_evaluate_model(model, name, epochs=50):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    test_accuracies = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for inputs, labels in train_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels.view(-1, 1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        correct_test = 0
        total_test = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                predicted = (outputs.data > 0.5).float()
                total_test += labels.size(0)
                correct_test += (predicted.view(-1) == labels).sum().item()

        avg_train_loss = train_loss / len(train_loader)
        test_accuracy = correct_test / total_test

        train_losses.append(avg_train_loss)
        test_accuracies.append(test_accuracy)

    model.eval()
    with torch.no_grad():
        y_pred_tensor = model(X_test_tensor)
        y_pred = (y_pred_tensor.numpy().flatten() > 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"Modelo: {name}")
    print(f"Precisión: {accuracy:.4f}")
    print(classification_report(y_test, y_pred))

    return accuracy, train_losses, test_accuracies

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.output_layer = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.output_layer(x)
        x = self.sigmoid(x)
        return x

In [ ]:
class DeepNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(DeepNN, self).__init__()
        self.layers = nn.ModuleList()

        self.layers.append(nn.Linear(input_size, hidden_sizes[0]))

        for i in range(len(hidden_sizes) - 1):
            self.layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))

        self.output_layer = nn.Linear(hidden_sizes[-1], output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            x = nn.functional.relu(x)
            if i < len(self.layers) - 1:
                x = nn.functional.dropout(x, p=0.2, training=self.training)

        x = self.output_layer(x)
        x = self.sigmoid(x)
        return x

In [ ]:
simple_model = SimpleNN(input_size, 64, output_size)
accuracy_simple, train_losses_simple, test_accuracies_simple = train_and_evaluate_model(
    simple_model, "Red Simple (1 capa oculta)"
)

deep_model = DeepNN(input_size, [64, 32, 16], output_size)
accuracy_deep, train_losses_deep, test_accuracies_deep = train_and_evaluate_model(
    deep_model, "Red Profunda (3 capas ocultas)"
)

In [ ]:
print("\nComparación de todos los modelos:")
print(f"Regresión Logística: {lr_accuracy:.4f}")
print(f"Red Neuronal Original (2 capas ocultas): {nn_accuracy:.4f}")
print(f"Red Simple (1 capa oculta): {accuracy_simple:.4f}")
print(f"Red Profunda (3 capas ocultas): {accuracy_deep:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
epochs_range = range(1, len(test_accuracies_simple) + 1)
plt.plot(epochs_range, test_accuracies_simple, label="Red Simple (1 capa)")
plt.plot(epochs_range, test_accuracies_deep, label="Red Profunda (3 capas)")
plt.xlabel("Épocas")
plt.ylabel("Precisión en Prueba")
plt.title("Comparacion de Arquitecturas de Red Neuronal")
plt.legend()
plt.show()
plt.close()

Comparacion del rendimiento con los resultados previos de regresion logistica:

    Precisión:

        Regresión Logistica: 0.9825

        Red Neuronal Original (2 capas ocultas): 0.9649

        Red Simple (1 capa oculta): 0.9561

        Red Profunda (3 capas ocultas): 0.9561

La regresion logistica obtuvo la mayor precision (98.25%), superando a todas las arquitecturas de redes neuronales probadas, Las redes neuronales, si bien fueron cercanas, no lograron superar este rendimiento, con precisiones que oscilaron entre un 95.61% y 96.49%

    Curvas ROC y AUC:

        Regresion Logistica (AUC = 0.995): La curva ROC de la regresion logistica mostro un area bajo la curva (AUC) ligeramente superior a la de la red neuronal

        Red Neuronal (AUC = 0.991): Aunque cercana, la red neuronal tuvo un AUC ligeramente inferior, lo que indica que la regresion logistica tuvo un mejor desempeño en la clasificacion.

    Matriz de Confusion:
        Ambos modelos (regresion logistica y red neuronal) mostraron un buen desempeño en la clasificacion de las clases, con pocos errores. Sin embargo, la regresion logistica tuvo un mejor equilibrio entre precision y recall, especialmente en la clase minoritaria (malignant).

Visualizacion del proceso de entrenamiento:

    Curvas de Perdida: Durante el entrenamiento, la perdida en el conjunto de entrenamiento disminuyo rapidamente, mientras que la pérdida en el conjunto de prueba se mantuvo relativamente estable, lo que indica que no hubo sobreajuste significativo.

    Curvas de Precision: La precision en el conjunto de entrenamiento alcanzo el 100% en las ultimas epocas, mientras que la precision en el conjunto de prueba se estabilizo alrededor del 96.49%, lo que sugiere que el modelo generalizo bien

Conclusión:

La regresion logistica demostro ser el modelo mas efectivo para este conjunto de datos, superando a las redes neuronales en terminos de precision y AUC. Aunque las redes neuronales tambien tuvieron un buen desempeño, no lograron superar a la regresion logistica